In [1]:
!date

Thu Sep 17 15:33:55 PDT 2026


In [2]:
import glob
import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

In [3]:
projdir = '/u/project/cluo/terencew/claude/project_ideas/asm_lr_hprc2'
outdir = f'{projdir}/results/qc/data'

### PCLAI local-ancestry PCs, every donor/haplotype (456 files, was 2 donors in the pilot version)

In [4]:
def load_pclai(args):
    sample, hap = args
    path = f'{projdir}/tsv/meta/pclai/{sample}_hap{hap}_pclai_v1.1.grch38_coord.bed'
    df = pd.read_csv(path, sep='\t', header=None)
    coords = df[9].str.strip('()').str.split(',', expand=True).astype(float)
    return {'sample': sample, 'hap': hap, 'pclai_pc1_mean': coords[0].mean(), 'pclai_pc2_mean': coords[1].mean(),
            'pclai_pc1_sd': coords[0].std(), 'pclai_pc2_sd': coords[1].std(), 'n_windows': len(df)}

samples = sorted({f.split('/')[-1].split('_hap')[0] for f in glob.glob(f'{projdir}/tsv/meta/pclai/*_hap1_*.bed')})
tasks = [(s, h) for s in samples for h in (1, 2)]
print(len(samples), 'donors,', len(tasks), 'haplotype files')

228 donors, 456 haplotype files


In [5]:
with ProcessPoolExecutor(10) as ex:
    pclai = pd.DataFrame(list(tqdm(ex.map(load_pclai, tasks), total=len(tasks))))

100%|██████████| 456/456 [00:11<00:00, 39.49it/s]


In [6]:
pclai.head()

,sample,hap,pclai_pc1_mean,pclai_pc2_mean,pclai_pc1_sd,pclai_pc2_sd,n_windows
0,HG00097,1,0.445048,-1.312791,0.001238,0.031211,26036
1,HG00097,2,0.444920,-1.311651,0.019253,0.044425,26031
2,HG00099,1,0.444147,-1.311559,0.044998,0.049224,26029
3,HG00099,2,0.445236,-1.308992,0.004479,0.068850,26028
4,HG00126,1,0.445187,-1.309284,0.002439,0.061511,26011


In [7]:
pclai.shape

(456, 7)

### sequencing covariates (HPRC2 Supp S6, all donors) + measured per-CpG depth (B01a, 402 haps)

In [8]:
seq = pd.read_csv(f'{projdir}/results/qc/data/supp_seq_qc.csv')[
    ['sample_id', 'sequencing_chemistry_ont', 'read_N50_ont', 'coverage_ont', 'coverage_100kb_ont']].rename(
    columns={'sample_id': 'sample'})
meas = pd.concat([pd.read_csv(f, sep='\t') for f in glob.glob(f'{projdir}/results/meth_bins/per_hap/*.summary.tsv')])[
    ['sample', 'hap', 'mean_depth', 'median_depth', 'n_cpg_native', 'n_cpg_hg38', 'global_wmeth']]
man = pd.read_csv(f'{projdir}/tsv/meta/hprc2_sample_manifest.tsv', sep='\t')[
    ['sample_id', 'population', 'superpopulation']].rename(columns={'sample_id': 'sample'})
merged = pclai.merge(meas, on=['sample', 'hap'], how='outer').merge(seq, on='sample', how='left').merge(man, on='sample', how='left')

In [9]:
merged.head()

,sample,hap,pclai_pc1_mean,pclai_pc2_mean,pclai_pc1_sd,pclai_pc2_sd,n_windows,mean_depth,median_depth,n_cpg_native,n_cpg_hg38,global_wmeth,sequencing_chemistry_ont,read_N50_ont,coverage_ont,coverage_100kb_ont,population,superpopulation
0,HG00097,1,0.445048,-1.312791,0.001238,0.031211,26036,25.394918,25.0,32408753.0,26498393.0,0.639523,R1041,89210.35,57.09,25.34,GBR,EUR
1,HG00097,2,0.444920,-1.311651,0.019253,0.044425,26031,25.418936,25.0,32023247.0,26461936.0,0.634447,R1041,89210.35,57.09,25.34,GBR,EUR
2,HG00099,1,0.444147,-1.311559,0.044998,0.049224,26029,27.427522,27.0,32420616.0,26448515.0,0.684083,R941,72139.93,58.78,20.75,GBR,EUR
3,HG00099,2,0.445236,-1.308992,0.004479,0.068850,26028,27.470538,27.0,32286272.0,26468116.0,0.678444,R941,72139.93,58.78,20.75,GBR,EUR
4,HG00126,1,0.445187,-1.309284,0.002439,0.061511,26011,33.891916,33.0,31775871.0,26469735.0,0.668717,R1041,107245.82,70.94,38.34,GBR,EUR


In [10]:
merged.shape

(456, 18)

In [11]:
merged.groupby('superpopulation')[['pclai_pc1_mean', 'pclai_pc2_mean', 'mean_depth', 'read_N50_ont']].median().round(3)

,pclai_pc1_mean,pclai_pc2_mean,mean_depth,read_N50_ont
superpopulation,,,,
AFR,-1.740,0.206,29.234,83485.20
AMR,0.404,-0.509,30.849,79821.65
EAS,0.694,0.904,31.596,81754.87
EUR,0.445,-1.306,29.625,79124.88
SAS,0.477,-0.511,28.926,79843.38


### does PCLAI agree with the 1000G-panel superpopulation label?

In [12]:
print(merged.groupby('superpopulation')[['pclai_pc1_mean', 'pclai_pc2_mean']].agg(['median', 'std']).round(3).to_string())
print('\nwithin-donor hap1 vs hap2 PCLAI PC1 correlation: %.4f' % merged.pivot_table(
    index='sample', columns='hap', values='pclai_pc1_mean').corr().iloc[0, 1])

                pclai_pc1_mean        pclai_pc2_mean       
                        median    std         median    std
superpopulation                                            
AFR                     -1.740  0.177          0.206  0.107
AMR                      0.404  0.305         -0.509  0.670
EAS                      0.694  0.002          0.904  0.013
EUR                      0.445  0.004         -1.306  0.007
SAS                      0.477  0.005         -0.511  0.039

within-donor hap1 vs hap2 PCLAI PC1 correlation: 0.9927


In [13]:
merged.to_csv(f'{outdir}/pclai_seq_covariates_per_hap.tsv', sep='\t', index=False)
print('wrote', f'{outdir}/pclai_seq_covariates_per_hap.tsv', merged.shape)

wrote /u/project/cluo/terencew/claude/project_ideas/asm_lr_hprc2/results/qc/data/pclai_seq_covariates_per_hap.tsv (456, 18)


In [14]:
!date

Thu Sep 17 15:34:10 PDT 2026
